# rl19_synth

> Phase 19 (`specs/multiagent/phase19/`): synthetic evaluation rows with known effects for the tests and the quick demo (`syntheticRows19`: every level of `contestSuite`, the sequences included), and the power check run before `RL19_PLAN` is frozen (`powerPrimary`: the two primary exact permutation tests under Holm, simulated with a per-run spread such as Phase 17's 8 runs; design-low-level, Statistics).

In [ ]:
#| default_exp rl19_synth

In [ ]:
#| hide
from nbdev.showdoc import *

## Synthetic rows

In [ ]:
#| export
import itertools
import json
from functools import lru_cache

import numpy as np

from explRL.bench_stats import holm
from explRL.rl19_grid import LIFETIMES, RUNS19
from explRL.rl19_suite import EVAL19_SEEDS, FAULTS19, GATE_LIARS, HELD_SIZES, K19, SEQ_SIZES
from explRL.rl19_step import SEQ


def _arm(m):
    return m.rsplit('_s', 1)[0] if m != 'warm' else 'warm'


def syntheticRows19(damage=0.25, slope=(0.04, 0.01), spec=True, seed=0, runs=RUNS19, seeds=EVAL19_SEEDS, cost=0.01):
    """Rows for the warm start and every run with known effects: arm 2 wins back `damage` more of the liars' damage than arm 1
    (arm 3 half that); arm 2's waiting rate rises by slope[0] per liar, arm 3's by slope[1] (arms 1, 4 and the warm start
    0.01), arm 2 waits 0.02 less at lifetime 15; arm 2 costs `cost` of honest pooled share without liars; with `spec`, arm 4's
    agents keep fixed roles over its sequences (the warm start's do not). Small run and board noise."""
    rng, rows = np.random.default_rng(seed), []
    D = dict(warm=0.3, a19credit=0.3, a19contest=0.3 + damage, a19fixed=0.3 + damage / 2, a19open=0.2)
    S = dict(a19contest=slope[0], a19fixed=slope[1])
    row = lambda m, t, N, sd, nl, **k: dict(level=f'{m}|{t}', seed=sd, N=N, nLiar=nl, tEnd=40.0, inputTokens=1000.0, outputTokens=100.0,
                                            misreport=1.0, sharePerHeld=0.5, contests=2.0 if nl else 0.0, liarPosts=4.0 if nl else 0.0,
                                            guardLies=2.0 if nl else 0.0, guardContests=1.0 if nl else 0.0,
                                            contestLags=json.dumps([1.0, 2.0] if nl else []), honestAnswers=10.0, honestTurns=480.0,
                                            misinformedFirst=2.0 if nl else 0.0, liarFirst=3.0 if nl else 0.0, uncorrected=1.0 if nl else 0.0,
                                            wrongHonest=4.0, sameWrong=1.0, **k)
    for m in ['warm'] + [f'{a}_s{s}' for a, s in runs]:
        a, u, e = _arm(m), rng.normal(0, 0.002), lambda: rng.normal(0, 0.003)
        for nl in GATE_LIARS:
            for L in LIFETIMES:
                wr = 0.2 + S.get(a, 0.01) * nl - (0.02 if a == 'a19contest' and L == 15 else 0) + u
                for sd in seeds:
                    wh = 0.40 - 0.025 * nl * (1 - D[a]) + u + e()
                    hp = (0.5 - (cost if a == 'a19contest' else 0)) if nl == 0 else 0.4
                    rows.append(row(m, f'gL{nl}T{L}', 16, sd, nl, welfareHonest=wh, welfare=wh, honestPooled=hp + e(),
                                    usedAnswers=50.0, waitAnswers=float(round(50 * wr))))
                    if m == 'warm':
                        pw = 0.40 - 0.025 * nl + e()
                        rows.append(row(m, f'plaingL{nl}T{L}', 16, sd, nl, welfareHonest=pw, welfare=pw, honestPooled=0.5 if nl == 0 else 0.4))
                        rows += [row(m, f'k{K}L{nl}T{L}', 16, sd, nl, welfareHonest=0.37 + 0.01 * K - 0.02 * nl + e()) for K in K19]
        held = [(f'n{N}L{n}', N, n) for N in HELD_SIZES for n in (0, N // 4)] + [(f, 16, 4) for f in FAULTS19]
        for t, N, nl in held:
            for sd in seeds:
                rows.append(row(m, t, N, sd, nl, welfareHonest=0.40 - 0.1 * bool(nl) * (1 - D[a]) + e()))
                if m == 'warm':
                    rows.append(row(m, 'plain' + t, N, sd, nl, welfareHonest=0.40 - 0.1 * bool(nl) + e()))
        if a in ('a19open', 'warm'):
            rows += _seqRows(m, rng, spec and a == 'a19open', seeds)
    return rows


def _seqRows(m, rng, fixed, seeds):
    out = []
    for N in SEQ_SIZES:
        nl = N // 4
        for kind in ('seq',) + (('seqrm',) if N == 16 else ()):
            for sd in seeds:
                p_ = {i: (0.6 if i % 2 else 0.1) if fixed else 0.35 for i in range(N)}
                for pos in range(SEQ):
                    ag = [i for i in range(N) if not (kind == 'seqrm' and pos in (3, 4) and nl <= i < nl + N // 4)]
                    sh = [0 if i < nl else int(rng.binomial(40, p_[i])) for i in ag]
                    out.append(dict(level=f'{m}|{kind}{N}', seed=10 * sd + pos, N=len(ag), nLiar=nl, seqId=sd, pos=pos, agents=json.dumps(ag),
                                    agentTurns=json.dumps([40] * len(ag)), agentShares=json.dumps(sh),
                                    agentContests=json.dumps([int(rng.binomial(40, 0.05)) if i >= nl else 0 for i in ag]),
                                    agentSubmits=json.dumps([(40 - x) // 2 for x in sh]), agentOpens=json.dumps([(40 - x) // 4 for x in sh])))
    return out

## The power check

In [ ]:
#| export
@lru_cache(maxsize=None)
def _splits(n, k):
    return np.array(list(itertools.combinations(range(n), k)))


def permP(a, b):
    "rl16_report.permTest's one-sided exact p, vectorised: the share of relabelings whose mean difference is at least the observed."
    pool, k = np.concatenate([a, b]).astype(float), len(a)
    if not np.isfinite(pool).all():
        raise ValueError('permP needs finite values')
    s = pool[_splits(len(pool), k)].sum(1)
    d = s / k - (pool.sum() - s) / (len(pool) - k)
    return float(np.mean(d >= d[0] - 1e-12))


def powerPrimary(effects, sds, n=(8, 8), nsim=500, alpha=0.05, seed=0):
    """Power of the primary family: per simulation, each test's arms draw per-run values (normal; arm A at the effect, arm B at
    0, the given per-run standard deviation, e.g. that of Phase 17's 8 runs' welfare gains), the exact one-sided permutation
    p, then Holm across the tests. -> {test: share of simulations it is rejected at alpha}."""
    rng, hs = np.random.default_rng(seed), list(effects)
    if not all(np.isfinite([effects[h], sds[h]]).all() and sds[h] > 0 for h in hs):
        raise ValueError(f'effects and positive standard deviations needed, got {effects}, {sds}')
    hit = dict.fromkeys(hs, 0)
    for _ in range(nsim):
        p = [permP(rng.normal(effects[h], sds[h], n[0]), rng.normal(0.0, sds[h], n[1])) for h in hs]
        for h, q in zip(hs, holm(p)):
            hit[h] += bool(q < alpha)
    return {h: hit[h] / nsim for h in hs}

## Tests

In [ ]:
# permP equals rl16_report.permTest; the power is high for a large effect, near alpha (one test of two under Holm: at most) for none
from explRL.rl16_report import permTest
from explRL.rl19_synth import *
r = np.random.default_rng(3)
for _ in range(3):
    a, b = r.normal(0.1, 1, 8), r.normal(0, 1, 8)
    assert abs(permP(a, b) - permTest(list(a), list(b))['p']) < 1e-12
assert permP(np.arange(8) + 10.0, np.arange(8.0)) == 1 / 12870
for bad in ((permP, (np.array([np.nan] * 8), np.zeros(8))), (powerPrimary, (dict(H19_1=0.1), dict(H19_1=float('nan'))))):
    try:
        bad[0](*bad[1])
        raise AssertionError(bad)
    except ValueError:
        pass
pw = powerPrimary(dict(H19_1=3.0, H19_2=0.0), dict(H19_1=1.0, H19_2=1.0), nsim=200)
assert pw['H19_1'] > 0.95 and pw['H19_2'] < 0.08, pw
rows = syntheticRows19(seeds=[900, 901])
assert {r['level'] for r in rows} >= {'warm|k3L8T30', 'a19open_s5|seqrm16', 'a19fixed_s8|collude', 'warm|plainn32L8'}
assert len([r for r in rows if r['level'] == 'a19open_s1|seqrm16']) == 12 and not any(r['level'].startswith('a19credit_s1|seq') for r in rows)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()